In [ ]:
;; Função auxiliar para buscar a transição correta na lista
(define (encontrar-transicao estado evento transicoes)
  (cond ((null? transicoes) #f)
        ((and (eq? (caar transicoes) estado)
              (eq? (cadar transicoes) evento))
         (car transicoes))
        (else (encontrar-transicao estado evento (cdr transicoes)))))

;; Motor que cria a closure da máquina
(define (criar-maquina lista-estados lista-transicoes)
  (let* ((estado-inicial (car lista-estados)) ; O primeiro vira inicial por padrão
         (estado-atual-val estado-inicial))   ; Memória interna da máquina
    
    ;; A máquina retornada é uma função que escuta mensagens
    (lambda (mensagem . argumentos)
      (cond
        [(eq? mensagem 'estado-atual) 
         estado-atual-val]
        
        [(eq? mensagem 'definir-inicio)
         (set! estado-inicial (car argumentos))
         (set! estado-atual-val estado-inicial)
         estado-inicial]
        
        [(eq? mensagem 'resetar)
         (set! estado-atual-val estado-inicial)
         estado-atual-val]
        
        [(eq? mensagem 'passo)
         (let* ((evento (car argumentos))
                (trans (encontrar-transicao estado-atual-val evento lista-transicoes)))
           (if trans
               (begin
                 (set! estado-atual-val (caddr trans)) ; Atualiza o estado
                 estado-atual-val)                     ; Retorna novo estado
               (display "Erro: Transicao invalida para este estado!")))]
               
        [(eq? mensagem 'investigar)
         (let* ((estado-teste (car argumentos))
                (evento-teste (cadr argumentos))
                (trans (encontrar-transicao estado-teste evento-teste lista-transicoes)))
           (if trans
               (caddr trans)
               (display "Erro: Transicao nao encontrada!")))]))))

;; Funções da linguagem FSML
(define (estado-atual maquina) 
  (maquina 'estado-atual))

(define (definir-inicio maquina estado) 
  (maquina 'definir-inicio estado))

(define (resetar maquina) 
  (maquina 'resetar))

(define (passo maquina evento) 
  (maquina 'passo evento))

(define (investigar maquina estado evento) 
  (maquina 'investigar estado evento))

(define-syntax definir-maquina
  (syntax-rules (estados transicoes PARA COM)
    [(_ nome-da-maquina
        (estados estado ...)
        (transicoes (origem PARA destino COM evento) ...))
     
     ;; O macro expande a sintaxe bonita para isto:
     (define nome-da-maquina
       (criar-maquina '(estado ...) 
                      '((origem evento destino) ...)))]))

# Tutorial e Exemplos

## Criando uma Máquina

Para criar uma nova máquina de estados, utilizamos o comando definir-maquina. A sintaxe exige que você defina um nome para a máquina e, em seguida, forneça dois blocos obrigatórios:

(estados ...): Liste todos os estados possíveis. O primeiro estado digitado será automaticamente o estado inicial da máquina.

(transicoes ...): Escreva as regras de mudança. A estrutura de cada regra deve ser exatamente: (estado-de-origem PARA estado-de-destino COM nome-do-evento).

Exemplo Prático: Vamos criar a máquina mario simulando os movimentos de um personagem.

In [ ]:
;;Máquina de estados baseada no diagrama de estados de uma task em RTOS
(definir-maquina tarefa-so
  (estados ready running blocked)
  (transicoes
    (ready PARA running COM obter_prioridade)
    (running PARA ready COM perder_prioridade)
    (running PARA blocked COM aguardar_recurso)
    (blocked PARA ready COM desbloqueio_normal)
    (blocked PARA running COM desbloqueio_prioritario)))

(display "Maquina 'tarefa-so' criada com sucesso!")

Maquina 'tarefa-so' criada com sucesso!

## Consultando o Estado Atual

Se você precisa saber em qual estado a máquina se encontra neste exato momento, utilize a função estado-atual.

Como usar: Digite (estado-atual nome-da-maquina).

Ele retornará um símbolo (com um apóstrofo na frente) representando a situação real do seu autômato, sem alterar nada.

In [ ]:
(display "Estado inicial da tarefa (apos inicializacao): ")
(estado-atual tarefa-so)
;;Retorna: 'ready

Estado inicial da tarefa (apos inicializacao): blocked

## Se movimentando dentro da máquina de estados através do "Passo"

Para fazer a máquina "andar", ou seja, transitar de um estado para o outro, utilizamos a função passo. Você deve informar qual evento está acontecendo.

Como usar: Digite (passo nome-da-maquina 'nome-do-evento). Atenção: o evento deve levar um apóstrofo ' antes do nome para que o Scheme o entenda como um símbolo.

Se a transição for válida, a máquina muda de estado permanentemente e retorna o novo estado. Se for inválida, exibirá um erro.

In [11]:
(display "O escalonador deu a maior prioridade para a tarefa: ")
(passo tarefa-so 'obter_prioridade)
;; Retorna: 'running

(display "\nA tarefa tentou acessar um semaforo ocupado: ")
(passo tarefa-so 'aguardar_recurso)
;; Retorna: 'blocked

O escalonador deu a maior prioridade para a tarefa: Erro: Transicao invalida para este estado!
A tarefa tentou acessar um semaforo ocupado: Erro: Transicao invalida para este estado!

## Simulando cenários através do "Investigar"

E se você quiser saber o que aconteceria caso um evento ocorresse, mas sem de fato alterar a máquina? Use a função investigar.

Como usar: Digite (investigar nome-da-maquina 'estado-hipotetico 'evento-hipotetico). Lembre-se dos apóstrofos nos parâmetros!

Essa função consulta as regras que você criou lá em cima e retorna o destino da transição, sem mexer no estado real atual da máquina.

In [ ]:
(display "Simulacao: Se a tarefa estivesse 'blocked' e sofresse um 'desbloqueio_normal', para onde iria? \nResultado: ")
(investigar tarefa-so 'blocked 'desbloqueio_normal)
;; Retorna: 'ready

(display "\nValidando que o estado real continua o mesmo da celula anterior: ")
(estado-atual tarefa-so)
;; Retorna: 'blocked

Simulacao: Se a tarefa estivesse 'blocked' e sofresse um 'desbloqueio_normal', para onde iria? 
Resultado: 
Validando que o estado real continua o mesmo da celula anterior: blocked

## Mudando o ponto de partida e resetando através do "resetar" e "definir-inicio"

Durante o uso ou testes, você pode precisar "reiniciar o jogo".

(resetar nome-da-maquina): Devolve a máquina instantaneamente para o seu estado inicial (que definimos lá na criação).

(definir-inicio nome-da-maquina 'novo-estado): Muda definitivamente qual é o estado padrão daquela máquina. A partir desse comando, o "ponto zero" passa a ser o novo estado escolhido e a máquina é movida para ele.

In [ ]:
(display "Resolvendo a pendencia e mandando a tarefa direto pra CPU: ")
(passo tarefa-so 'desbloqueio_prioritario) ; Retorna para 'running

(display "\nDefinindo que esta tarefa agora sempre nascera bloqueada: ")
(definir-inicio tarefa-so 'blocked)

(display "\nReiniciando a tarefa (Reset): ")
(resetar tarefa-so)
;; Retorna 'blocked

Resolvendo a pendencia e mandando a tarefa direto pra CPU: 
Definindo que esta tarefa agora sempre nascera bloqueada: 
Reiniciando a tarefa (Reset): blocked